In [2]:
import pandas as pd

In [17]:
result_dir = "../results/processedData/normalized_mean/"

# produced by dropseq_final_atlas_analysis.R
gene_counts_normalized = "../results/processedData/genes_to_cell_type_table.tsf"
# produced by "uknown"
annotation_file = "../data/hvaepLRv2_kegg_go.tsv"

gene_counts_normalized_table = pd.read_table(gene_counts_normalized)
annotation_table = pd.read_table("../results/processedData/hvaep_uniprot_kegg_go.tsf")
annotation_table.columns = ["ID","SP","UniProt","KEGG","KO","GO"]

In [18]:
gene_counts_normalized_table.head()

,Ec_Head,En_Foot,En_BodyCol/SC,En_Head,Ec_BodyCol/SC,En_Tentacle,I_ZymoGl,I_ISC,Ec_Peduncle,Ec_Tentacle,...,I_DesmoNB,I_IsoNB,I_Ec1N,I_Ec3N,I_Ec4N,I_Ec1/5N,I_Ec2N,I_En2N,I_En1N,I_En3N
gfp,0.001744,0.002329,0.000435,0.001558,0.000299,0.002748,0.003147,0.032247,0.000000,0.004077,...,0.005979,0.013403,0.441901,0.256747,0.494403,0.287279,0.365067,0.282429,0.510763,0.060925
HVAEP1-G000002,0.019181,0.010354,0.009107,0.013010,0.026924,0.009749,0.004721,0.013623,0.020603,0.016309,...,0.006423,0.008779,0.000000,0.005498,0.004151,0.004332,0.008241,0.000000,0.004705,0.003431
HVAEP1-G000005,0.000000,0.000000,0.000000,0.000000,0.000299,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.001819,0.000000,0.000000
HVAEP1-G000008,0.015694,0.008538,0.009710,0.011452,0.014710,0.004252,0.009441,0.008433,0.016438,0.010193,...,0.004686,0.001915,0.007963,0.003598,0.002075,0.004332,0.004530,0.003639,0.000000,0.003431
HVAEP1-G000009,0.020925,0.005887,0.008000,0.017582,0.029543,0.014001,0.003934,0.015569,0.026490,0.012232,...,0.002454,0.003830,0.007702,0.008598,0.005365,0.004332,0.005432,0.006522,0.001568,0.003431


In [20]:
annotation_table.head()

,ID,SP,UniProt,KEGG,KO,GO
0,HVAEP1-G000002,P54310,LIPS_MOUSE,mmu:16890;,NaN,caveola [GO:0005901]; cytoplasm [GO:0005737]; ...
1,HVAEP1-G000009,P54310,LIPS_MOUSE,mmu:16890;,NaN,caveola [GO:0005901]; cytoplasm [GO:0005737]; ...
2,HVAEP1-G000010,O42342,SOX7_XENLA,xla:378665;,NaN,nucleus [GO:0005634]; sequence-specific DNA bi...
3,HVAEP1-G000012,Q15291,RBBP5_HUMAN,hsa:5929;,NaN,histone methyltransferase complex [GO:0035097]...
4,HVAEP1-G000022,Q60855,RIPK1_MOUSE,mmu:19766;,NaN,death-inducing signaling complex [GO:0031264];...


In [22]:
# execute this section if there is still the transcriptome id in use (marked by the "T" in the identifier)
# t_to_g_id = lambda x: "HVAEP1-" + "G"+ x.split(".T")[1].split(".")[0]
# annotation_table.ID = annotation_table.ID.apply(t_to_g_id)
# annotation_table.head()

In [23]:
annotation_table.to_csv(result_dir + "/hvaep_uniprot_kegg_go.tsf", sep="\t", index=False)

In [24]:
print("Length: {}, Length without duplicates: {}".format(len(annotation_table), len(annotation_table['ID'].unique())))

Length: 11826, Length without duplicates: 11826


In [25]:
print("Length: {}, Length without duplicates: {}".format(len(gene_counts_normalized_table.index), len(gene_counts_normalized_table.index.unique())))

Length: 18323, Length without duplicates: 18323


In [26]:
gene_counts_normalized_table["ID"] = gene_counts_normalized_table.index
normalized_gene_data = gene_counts_normalized_table.merge(annotation_table, on="ID")

# combine both dataframes
merged_df = pd.merge(annotation_table,gene_counts_normalized_table, left_on=["ID"], 
               right_on=["ID"],
               how='outer', indicator=True)
# 
combined_gene_table = pd.concat([merged_df.query('_merge == "right_only"'),normalized_gene_data])
combined_gene_table = combined_gene_table.sort_values(by='ID')

cols = list(gene_counts_normalized_table.columns)
cols.remove("ID")
cols.append("UniProt")
#Cluster ID	Gene ID|Swissprot Annotationb
with open(result_dir + "hvaep_cell_type_to_gene_cluster_table.tsf", 'w') as output_file:
    output_file.write("Cluster ID\tGene ID|Swissprot Annotation\n")
    for identifier in combined_gene_table.ID:
        temp_df = combined_gene_table[combined_gene_table.ID == identifier][cols]
        tdf = temp_df.drop("UniProt", axis=1)
        mean_for_outsort = tdf.T.mean()
        
        for col in temp_df:
            if col != "UniProt":
                # apply filter to discbard low read counts ? 
                if temp_df[col].values[0] > mean_for_outsort.values[0]:
                #if temp_df[col].values[0] > 1:
                    if type(temp_df["UniProt"].values[0]) == str:
                        output_file.write(col + "\t" + identifier + "|"+ temp_df["UniProt"].values[0] + "\n")
                    else:
                        output_file.write(col+ "\t" + identifier + "\n") 


In [30]:
gene_to_celltype_table = pd.read_table(result_dir + "hvaep_cell_type_to_gene_cluster_table.tsf")
gene_to_celltype_table.head()

,Cluster ID,Gene ID|Swissprot Annotation
0,Ec_Head,HVAEP1-G000002|LIPS_MOUSE
1,En_Head,HVAEP1-G000002|LIPS_MOUSE
2,Ec_BodyCol/SC,HVAEP1-G000002|LIPS_MOUSE
3,I_ISC,HVAEP1-G000002|LIPS_MOUSE
4,Ec_Peduncle,HVAEP1-G000002|LIPS_MOUSE


In [31]:
gene_to_celltype_table = pd.read_table(result_dir + "hvaep_cell_type_to_gene_cluster_table.tsf")
print("[+] Length cell type to gene cluster table after filtering for mean values: {}".format(len(gene_to_celltype_table["Gene ID|Swissprot Annotation"].drop_duplicates())))

[+] Length cell type to gene cluster table after filtering for mean values: 18322


In [38]:
gene_counts_normalized_table = pd.read_table(gene_counts_normalized)

gene_counts_normalized_table["ID"] = gene_counts_normalized_table.index
normalized_gene_data = gene_counts_normalized_table.merge(annotation_table, on="ID")

# combine both dataframes
merged_df = pd.merge(annotation_table,gene_counts_normalized_table, left_on=["ID"], 
               right_on=["ID"],
               how='outer', indicator=True)
# 
combined_gene_table = pd.concat([merged_df.query('_merge == "right_only"'),normalized_gene_data])
combined_gene_table = combined_gene_table.sort_values(by='ID')

parsed_ids = []
cols = list(gene_counts_normalized_table.columns)
cols.remove("ID")
cols.append("UniProt")
cols.append("GO")
#Cluster ID	Gene ID|Swissprot Annotation
with open(result_dir + "hvaep_cell_type_to_gene_cluster_table_just_gos.tsf", 'w') as output_file:
    with open(result_dir + "hvaep_uniprot_go_cleaned_table.tsf",'w') as outfile:
        outfile.write("ID\tUniProt\tGO\n")
        output_file.write("Cluster ID\tGene ID|Swissprot Annotation\n")
        for identifier in combined_gene_table.ID:
            temp_df = combined_gene_table[combined_gene_table.ID == identifier][cols]
            tdf = temp_df.drop("UniProt", axis=1)
            tdf = tdf.drop("GO", axis=1)
            mean_for_outsort = tdf.T.mean()
            
            for col in temp_df:
                if col != "UniProt" and col != "GO":
                    # apply filter to discard low read counts ? 
                    if temp_df[col].values[0] > mean_for_outsort.values[0]:
                        if type(temp_df["UniProt"].values[0]) == str and type(temp_df['GO'].values[0]) == str:
                            output_file.write(col + "\t" + identifier + "|"+ temp_df["UniProt"].values[0] + "\n")
                            if identifier not in parsed_ids:
                                outfile.write(identifier+"\t"+temp_df.UniProt.values[0]+"\t"+temp_df.GO.values[0]+"\n")
                                parsed_ids.append(identifier)

In [43]:
cleaned_annotation_file = pd.read_csv(result_dir + "hvaep_uniprot_go_cleaned_table.tsf", sep="\t")
cleaned_cluster_file = pd.read_csv(result_dir + "hvaep_cell_type_to_gene_cluster_table_just_gos.tsf", sep="\t")

In [44]:
print("Length annotation table: {}".format(len(cleaned_annotation_file)))

Length annotation table: 10433


In [45]:
print("Length of output table: {}".format(len(cleaned_cluster_file)))

Length of output table: 104901


In [47]:
gene_counts_normalized_table = pd.read_table(gene_counts_normalized)

gene_counts_normalized_table["ID"] = gene_counts_normalized_table.index
normalized_gene_data = gene_counts_normalized_table.merge(annotation_table, on="ID")

# combine both dataframes
merged_df = pd.merge(annotation_table,gene_counts_normalized_table, left_on=["ID"], 
               right_on=["ID"],
               how='outer', indicator=True)
# 
combined_gene_table = pd.concat([merged_df.query('_merge == "right_only"'),normalized_gene_data])
combined_gene_table = combined_gene_table.sort_values(by='ID')

with open(result_dir + "hvaep_uniprot_go_full_table.tsf",'w') as outfile:
    outfile.write("ID\tUniProt\tGO\n")
    for identifier in combined_gene_table.ID:
        temp_df = combined_gene_table[combined_gene_table.ID == identifier]
        if type(temp_df.UniProt.values[0]) == str and type(temp_df.GO.values[0]) == str:
            #ID 	SP 	UniProt 	KEGG 	KO 	GO
            outfile.write(temp_df.ID.values[0]+"\t"+temp_df.UniProt.values[0]+"\t"+temp_df.GO.values[0]+"\n")
        elif type(temp_df.UniProt.values[0]) != str and type(temp_df.GO.values[0]) == str:
            outfile.write(temp_df.ID.values[0]+"\t"+''+"\t"+temp_df.GO.values[0]+"\n")
        elif type(temp_df.UniProt.values[0]) == str and type(temp_df.GO.values[0]) != str:
            outfile.write(temp_df.ID.values[0]+"\t"+temp_df.UniProt.values[0]+"\t"+''+"\n")
        else:
            outfile.write(temp_df.ID.values[0]+"\t"+''+"\t"+''+"\n")

# Change deseq2 table header

## No need to execute for the current run -> analysis based on genes.results and not on isoforms.results

In [3]:
#df = pd.read_csv("../data/hydra_all_counts.tsv", sep="\t")
#df.head()
#df.target_id = df.target_id.apply(lambda x: "HVAEP1-" + x.split(".")[1].replace('T','G'))
#df.head()

In [11]:
df = pd.read_csv("../results/deseq2_rsem/hydra_all_counts.tsv", sep="\t")
df.gene_id = df.gene_id.apply(lambda x: "HVAEP1-" + x.split(".")[1])
df.to_csv("../results/deseq2_rsem/hydra_all_counts_t_to_g.tsv", sep="\t", header=True, index=False)

df = pd.read_csv("../results/deseq2_rsem/hydra_all_deseq2_lg2.tsv", sep="\t")
df.ID = df.ID.apply(lambda x: "HVAEP1-" + x.split(".")[1])
df.to_csv("../results/deseq2_rsem/hydra_all_deseq2_lg2_t_to_g.tsv", sep="\t", header=True, index=False)

df = pd.read_csv("../results/deseq2_rsem/hydra_all_deseq2_pvalues.tsv", sep="\t")
df.ID = df.ID.apply(lambda x: "HVAEP1-" + x.split(".")[1])
df.to_csv("../results/deseq2_rsem/hydra_all_deseq2_pvalues_t_to_g.tsv", sep="\t", header=True, index=False)
df.head()

,ID,EcoKD1_Eco1KD_B5_vs_control_B5,EcoKD1_Eco1KD_B8_vs_control_B8,HydraAHL_3OC12_vs_control,HydraAHL_3OHC12_vs_control,HydraRecolonization_Conventionalized_vs_GF,HydraRecolonization_Conventionalized_vs_Wild,HydraRecolonization_Cvbct_vs_GF,HydraRecolonization_Cvbct_vs_Wild,HydraRecolonization_GF_vs_Wild,HydraRecolonization_Wild_vs_GF,HydraTemperature_08°C_vs_18°C,HydraTemperature_12°C_vs_18°C,HydraTemperature_22°C_vs_18°C
0,HVAEP1-G011796,1.209753e-118,0.000000,2.465290e-02,5.792489e-03,0.016695,0.925976,0.009906,0.687587,0.000004,0.000004,3.952910e-116,7.106119e-108,1.617574e-04
1,HVAEP1-G011792,7.105124e-67,0.000143,9.047853e-06,3.093524e-03,0.237540,0.510849,0.336130,0.157256,0.000028,0.000028,3.524240e-20,3.575301e-21,3.673534e-10
2,HVAEP1-G008093,1.196686e-24,0.000006,4.062779e-04,3.668332e-01,0.062364,0.999464,0.223841,0.000074,0.003737,0.003737,9.010328e-12,3.125984e-15,2.941640e-01
3,HVAEP1-G028193,3.482068e-23,NaN,5.800923e-03,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,HVAEP1-G003526,2.003146e-21,0.856874,2.279078e-22,8.483718e-10,0.569012,0.919543,0.343898,0.898432,0.059384,0.059384,2.819526e-05,3.248345e-01,2.108662e-01


# Full Annotation Jay

In [ ]:
full_annotation_file = pd.read_csv("../data/annotation_KO_GO.csv")

In [ ]:
full_annotation_file.head()

In [ ]:
len(full_annotation_file)

In [ ]:
 'HVAEP1.T000001.1'.split(".")[1].replace("T","G")

# RSEM count files

In [ ]:
df = pd.read_csv("../results/deseq2_rsem/hydra_all_deseq2_lg2.tsv", sep="\t")
df.ID = df.ID.apply(lambda x: "HVAEP1-" + x.split(".")[1].replace('T','G'))
df.to_csv("../results/deseq2_rsem/hydra_all_deseq2_lg2_t_to_g.tsv", sep="\t", header=True, index=False)

In [ ]:
len(df["ID"])

In [ ]:
len(df["ID"].unique())

In [ ]:
df[df["ID"] == "HVAEP1-G028897"]

In [ ]:
df = pd.read_csv("../results/deseq2_rsem/hydra_all_deseq2_pvalues.tsv", sep="\t")
#df.ID = df.ID.apply(lambda x: "HVAEP1-" + x.split(".")[1].replace('T','G'))
#df.to_csv("../results/deseq2_rsem/hydra_all_deseq2_pvalues_t_to_g.tsv", sep="\t", header=True, index=False)

In [ ]:
len(df)

In [ ]:
len(df.ID.unique())

In [ ]:
df = pd.read_csv("../results/deseq2_rsem/hydra_all_counts.tsv", sep="\t")
#df.ID = df.ID.apply(lambda x: "HVAEP1-" + x.split(".")[1].replace('T','G'))
#df.to_csv("../results/deseq2_rsem/hydra_all_deseq2_pvalues_t_to_g.tsv", sep="\t", header=True, index=False)

In [ ]:
df.head()